In [1]:
%%capture
!pip install ibm-watsonx-ai==0.2.6
!pip install langchain==0.1.16
!pip install langchain-ibm==0.1.4
!pip install transformers==4.41.2
!pip install huggingface-hub==0.23.4
!pip install sentence-transformers==2.5.1
!pip install chromadb
!pip install wget==3.2
!pip install --upgrade torch --index-url https://download.pytorch.org/whl/cpu

In [5]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')
# Loaders and preprocessing
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter

# Embeddings and vector store
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma

# Prompt and chains
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

# Memory (optional for chat)
from langchain.memory import ConversationBufferMemory

from ibm_watsonx_ai.foundation_models import Model
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import ModelTypes, DecodingMethods
from ibm_watson_machine_learning.foundation_models.extensions.langchain import WatsonxLLM
import wget

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

# DATA/CORPUS.JSON

This notebook aimed at setting data/corpus.json. We webscrap from some url part 

In [2]:
filename = 'companyPolicies.txt'
url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/6JDbUb_L3egv_eOkouY71A.txt'
url_2 = "https://share.google/QbJCh2ygUQQBlGUtb"

text = wget.download(url)

100% [..............................................................................] 15660 / 15660

Autorisé : https://share.google/QbJCh2ygUQQBlGUtb
Autorisé : https://share.google/0sJ6jsMgez1sI3Ye5
Autorisé : https://afristat.org/wp-content/uploads/2022/04/NotesCours_Agri.pdf
Autorisé : https://afristat.org/wp-content/uploads/2022/04/NotesCours_Agri.pdf
Autorisé : https://share.google/JFsAeoTXBfuNtqTEu
Autorisé : https://docs.google.com/forms/d/e/1FAIpQLSduR9VwiyApSfLsXGEu6oklvzNXOkYF-S6m0IV1zu1P3TsDVw/viewform
Autorisé : https://doi.org/10.4060/cd3185en
Autorisé : https://doi.org/10.4060/cd4965en
Autorisé : https://doi.org/10.4060/cd4313en
Autorisé : https://doi.org/10.4060/cc8166en
Autorisé : https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://www.nitidae.org/files/acf8fe1c/manuel_de_formation_darda.pdf&ved=2ahUKEwipgaTzvdGQAxWBXEEAHaHjC5IQFnoECGsQAQ&sqi=2&usg=AOvVaw1byCUFVDKSINjUz5jsn5KD
Texte trop court ou illisible : https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://www.nitidae.org/files/acf8fe1c/manuel_de_formation_darda.pdf&ved=2

# Extracting existing documents


Texte trop court ou vide : A COMPETITIVE BOOK OF AGRICULTURE BY NEM RAJ SUNDA.pdf


Unable to create process using 'C:\Users\LENOVO T14s\anaconda3\python.exe "C:\Users\LENOVO T14s\anaconda3\Scripts\pip-script.py" install -U bitsandbytes'


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Define the BitsAndBytesConfig for 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config, # Use the BitsAndBytesConfig object
    trust_remote_code=True
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [3]:
from ollama import Client
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser,JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings


In [ ]:
llm = ChatOllama(model='llama3')
chain = promptZeroShot | llm | StrOutputParser 
responseChain = chain.invoke({'concept':'engrais'})